# Appendix D --- Reproducing the Artifacts

One command rebuilds everything: `python scripts/build_store.py` turns the corpus into the trained store (the only GPU step), and the `scripts/build_nb_*.py` builders regenerate every notebook. This appendix documents that pipeline and proves the *shipped* store reproduces the corpus facts byte-exact -- on CPU, no Qwen, so a reader can verify a fresh clone before spending a GPU minute.

In [ ]:
import os, sys
# Locate this topic's code/ dir (book_kit.py), robust to the working directory.
_cwd = os.getcwd()
for _p in (_cwd, os.path.join(os.path.dirname(_cwd), "code"), os.path.join(_cwd, "code")):
    if os.path.isfile(os.path.join(_p, "book_kit.py")):
        break
else:
    _p = os.environ.get("GMS_RAG_TUTORIAL", _cwd)
sys.path.insert(0, _p)
from book_kit import load_store, STORE

# Repo root for the artifact-tree listing below (absolute, cwd-independent).
REPO_ROOT = os.environ.get("GMS_RAG_TUTORIAL", "/path/to/forgeloop/beyond-chunk-and-pray/code")

## The artifact tree

In [ ]:
# The reproducible artifact tree. Everything a reader needs to rebuild the book
# is checked in; nothing here is generated by hand.
#
#   beyond-chunk-and-pray/
#     book/                          # the LaTeX manuscript
#     notebooks/                     # one runnable .ipynb per chapter
#     code/                          # REPO_ROOT below: all code + data live here
#       pyproject.toml               # deps: knowlytix, torch, transformers, pandas, jupyter
#       book_kit.py                  # shared bootstrap: repo paths + store loader
#       scripts/
#         _bootstrap.py              # makes `knowlytix` resolve to the branch source
#         build_store.py             # corpus  -> trained store + data/corpus_facts.md
#         build_nb_*.py              # one builder per chapter -> notebooks/*.ipynb
#         build_nb_appendix_D.py     # this appendix's builder
#       data/
#         annual_report.md           # the corpus (source of truth for content)
#         corpus_facts.md            # ground-truth facts sheet (generated)
#         eval_cohort.json           # labeled evaluation questions
#         gms_annual_report_store/   # the trained GMS store (generated)
#
# Two kinds of files live in code/data/: hand-authored inputs (annual_report.md,
# eval_cohort.json) and generated outputs (corpus_facts.md, the store). Generated
# outputs are committed so the book is runnable from a fresh clone without a GPU;
# build_store.py regenerates them byte-for-byte. The notebooks run from notebooks/,
# so REPO_ROOT resolves to the sibling code/ directory.
for sub in ("data/annual_report.md", "data/eval_cohort.json",
            "data/gms_annual_report_store", "scripts/build_store.py"):
    path = os.path.join(REPO_ROOT, sub)
    print(f"{'OK ' if os.path.exists(path) else 'MISSING':8} {sub}")

## The trained store on disk

In [ ]:
# The trained store is a directory of small artifacts -- a model checkpoint, an
# adapter, the ENM table, and the source markdown. It is portable: copy the
# directory and the store loads anywhere, no retraining. This is what build_store.py
# writes and what every chapter reloads.
for name in sorted(os.listdir(STORE)):
    print(name)

## The one GPU step: `build_store.py`

In [ ]:
# scripts/build_store.py is the single command that turns the corpus into the
# trained store. It is the only GPU step in the whole book. The configuration
# below is copied verbatim from that script -- it is the reproducible recipe:
# a fixed geometry, a fixed training budget, and regex ingest (deterministic,
# no LLM in the extraction path). Running it twice produces the same store.
#
#   from knowlytix.core.config import GeometryConfig, TrainConfig
#   from knowlytix.knowledge.config import DocGMSConfig
#   from knowlytix.knowledge.geode.rag import build_rag_store
#   from knowlytix.knowledge.geode.loop import make_default_trainer
#
#   cfg = DocGMSConfig(
#       store_path="data/gms_annual_report_store",
#       ingest_mode="regex",                 # deterministic extraction
#       loss_mode="cap",
#       geometry=GeometryConfig(d_v=64, d_u=64, m=32, d=32),
#       train=TrainConfig(epochs=150, batch_size=64, neg_samples=16,
#                         lr=5e-3, lr_riemannian=2e-3),
#   )
#   res = build_rag_store("data/annual_report.md", cfg, device=dev,
#                         geode_trainer=make_default_trainer(dev, epochs=80))
#
# res is a RagBuildResult with: converged, iterations, n_entities, n_triples,
# n_enm, corrections, anchor_violations. build_store.py prints those, then
# regenerates data/corpus_facts.md from the trained store. DO NOT run this cell
# in a shared-GPU session; the lead runs `python scripts/build_store.py` in CI.
print("build_store.py is the GPU step; see scripts/build_store.py. Skipped here.")

## Notebooks are built, not hand-edited

In [ ]:
# Notebooks are *built*, not edited by hand. Each scripts/build_nb_*.py emits one
# notebook with nbformat so the JSON is always valid and the first cell is always
# the exact KNOWLYTIX_SRC bootstrap. Re-running a builder regenerates its notebook
# deterministically; that is how a .tex listing stays byte-identical to the cell
# it documents. List the builders and the notebooks they own.
SCRIPTS = os.path.join(REPO_ROOT, "scripts")
builders = sorted(f for f in os.listdir(SCRIPTS)
                  if f.startswith(("build_nb", "build_notebook")))
print(f"{len(builders)} notebook builders:")
for b in builders:
    print(" ", b)

## Verifying the reproduction (CPU only)

In [ ]:
# The reproducibility claim: a fresh clone's *shipped* store reproduces the
# corpus facts byte-exact. We load the store on CPU (cheap -- no GPU, no Qwen)
# via book_kit.load_store(), which rebuilds the model with the exact build-time
# geometry/cap (GeometryConfig(64,64,32,32), loss_mode="cap"); a hand-built
# DocGMSConfig with default geometry would fail to load the state_dict. We then
# read three ENM figures back through lookup_enm. These are the exact values in
# data/corpus_facts.md; if a rebuild ever drifts, this cell fails first.
import torch

store = load_store(device=torch.device("cpu"))

# (category, entity_id, expected) -- verbatim from data/corpus_facts.md.
EXPECTED_ENM = [
    ("income_statement", "Revenue/FY2025", 355.0),
    ("segment_performance", "Cloud Platform/Technology/Revenue", 120.0),
    ("balance_sheet", "Total Assets", 540.0),
]
for category, entity_id, expected in EXPECTED_ENM:
    got = store.lookup_enm(category, entity_id)
    print(f"{category}/{entity_id} = {got}  (expected {expected})")
    assert got == expected, f"ENM drift: {category}/{entity_id} {got} != {expected}"
print("\nstore reproduces the shipped corpus facts byte-exact")

In [ ]:
# A second reproduction invariant, independent of any single value: the segment
# revenues must sum to the Total row. This is the numeric *sum anchor* GEODE
# enforces during the build (Chapter 5); checking it here proves the shipped
# store is internally consistent, not just that one cell happens to match.
SEGMENTS = ["Cloud Platform/Technology", "Devices/Technology",
            "Logistics/Operations", "Retail/Operations"]
parts = [store.lookup_enm("segment_performance", f"{s}/Revenue") for s in SEGMENTS]
total = store.lookup_enm("segment_performance", "Total/All/Revenue")
print("segments:", parts, " sum:", sum(parts), " total:", total)
assert sum(parts) == total, f"sum anchor broken: {sum(parts)} != {total}"
print("sum anchor holds: Total = sum(segments)")

## Exercise (worked solution)

In [ ]:
# Exercise (worked): strengthen the reproduction check with a second sum anchor
# and a two-hop retrieval -- both CPU-only, both grounded in data/corpus_facts.md.
# (1) headcounts must sum to the Total row.
head_parts = [store.lookup_enm("segment_performance", f"{s}/Headcount")
              for s in SEGMENTS]
head_total = store.lookup_enm("segment_performance", "Total/All/Headcount")
assert sum(head_parts) == head_total == 1500.0
print("headcount anchor holds:", sum(head_parts), "==", head_total)

# (2) reproduce the two-hop retrieval: cloud platform -> division -> region.
from knowlytix.knowledge.geode.provenance import ProvenanceLedger
from knowlytix.knowledge.rag import Retriever, TripleBinder
from knowlytix.knowledge.rag.query_triples import QueryTriple

ledger = ProvenanceLedger.from_text(store.markdown)
binder = TripleBinder(store)
retriever = Retriever(store, ledger)
chain = [binder.bind(QueryTriple("cloud platform", "has_division", "?x")),
         binder.bind(QueryTriple("?x", "has_region", "?"))]
result = retriever.retrieve(chain)
print("two-hop answers:", result.answers)
assert any(a[0] == "north america" for a in result.answers)
print("two-hop retrieval reproduces: north america")

## Self-check

In [ ]:
# Self-check: the chapter's claim is that a fresh clone reproduces the artifacts.
# We assert the artifact tree is present, the store loads, every documented ENM
# value reads back exactly, and the sum anchor holds -- the same gates CI runs.
assert os.path.exists(STORE), "store directory missing"
assert store.lookup_enm("income_statement", "Revenue/FY2025") == 355.0
assert sum(store.lookup_enm("segment_performance", f"{s}/Revenue")
           for s in SEGMENTS) == store.lookup_enm("segment_performance",
                                                   "Total/All/Revenue")
print("App D self-check passed: artifacts reproduce byte-exact from a clone.")